# ViennaRNA обзор

Python биндинги для доступных компонентов:

In [1]:
import RNA

Исходная последовательность для экспериментов:

In [2]:
sequence = "GGGGAAAACCCC"

## Компоненты, доступные из python

Сюда входят инструменты, вызываемые прямо из python:
- RNAfold
- RNAsubopt
- RNAeval

### RNAfold
Считает вариант сворачивания с минимальной свободной энергией (MFE), возвращает скобочную нотацию сворачивания и минимальную энергию.

In [3]:
folding, energy = RNA.fold(sequence)

print("Folding:", folding)
print("MFE:", energy)

Folding: ((((....))))
MFE: -5.400000095367432


Так как многие инструменты для работы требуют промежуточные матрицы энергий, логичнее использовать `fold_compaund` в работе.

In [4]:
fc = RNA.fold_compound(sequence)

### RNAsubopt

Выдаёт список субоптимальных структур и их энергий.

In [5]:
md = RNA.md()
md.uniq_ML = 1

fc = RNA.fold_compound(sequence, md)

# Enumerate suboptimal structures within 5 kcal/mol
subopts = fc.subopt(500)

for item in subopts:
    print(item.structure, item.energy)

((((....)))) -5.400000095367432
(((.....))). -4.199999809265137
.(((....))). -3.0999999046325684
(((......))) -2.700000047683716
.(((.....))) -2.4000000953674316
((......)).. -1.2000000476837158
.((.....)).. -1.100000023841858
.((......)). -0.4000000059604645


Для удобного использования результатов в дальнейшем коде имеет смысл использовать колбэк для аккумуляцмм результата в нужном формает.

In [6]:
results = []

def collect(structure, energy, data):
    if structure is not None:
        data.append({
            "structure": structure,
            "energy": energy
        })

fc.subopt_cb(500, collect, results)

print(results)

[{'structure': '(((......)))', 'energy': -2.700000047683716}, {'structure': '((((....))))', 'energy': -5.400000095367432}, {'structure': '.(((.....)))', 'energy': -2.4000000953674316}, {'structure': '(((.....))).', 'energy': -4.199999809265137}, {'structure': '.((......)).', 'energy': -0.4000000059604645}, {'structure': '.(((....))).', 'energy': -3.0999999046325684}, {'structure': '((......))..', 'energy': -1.2000000476837158}, {'structure': '.((.....))..', 'energy': -1.100000023841858}]


### RNAeval

Позволяет рассчитать MFE для заданной последовательности и структуры.

In [7]:
structure = "((((....))))"

fc = RNA.fold_compound(sequence)

energy = fc.eval_structure(structure)

print("Energy:", energy)

Energy: -5.400000095367432


In [8]:
fc.eval_structure_verbose(structure)

External loop                           :     0
Interior loop (  1, 12) GC; (  2, 11) GC:  -330
Interior loop (  2, 11) GC; (  3, 10) GC:  -330
Interior loop (  3, 10) GC; (  4,  9) GC:  -330
Hairpin  loop (  4,  9) GC              :   450


-5.400000095367432

## Недоступные пакеты

В этот раздел входят пакеты без доступа через python API, для использования этих инструментов нужны бинарники на хосте и их вызов через `subprocess`:
- kinfold
- barriers
- treekin